In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['CRYPTOGRAPHY_OPENSSL_NO_LEGACY'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import datasets
from datasets import load_dataset, Dataset, DatasetDict, Dataset
from tqdm.auto import tqdm
import json

datasets.disable_caching()

/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/miniconda3/envs/megatron-next/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from megatron.energon import get_train_dataset, WorkerConfig, get_loader, DefaultTaskEncoder, Cooker, basic_sample_keys
from megatron.energon import DefaultTaskEncoder, TextSample, Sample
import gzip
import pickle
import webdataset as wds
import json
import dataclasses
import torch

from tqdm.auto import tqdm
from typing import List

from qwen3_base_energon_helpers import MyTaskEncoder, print_error_handler

In [3]:
tokenizer_path="/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/code/Megatron-Next/model_dir/pulse_bench_v1_80b_a3b_next_gemini_bf16/tokenizer_v18_gemini"

In [4]:
worker_config = WorkerConfig(
    rank=0,
    world_size=1,
    seed_offset=123456789,
    num_workers=1,
)

train_ds = get_train_dataset(
    '/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/code/Megatron-Next/model_dir/pulse_bench_v1_80b_a3b_next_gemini_bf16/dataset.yaml',
    split_part="train",
    task_encoder=MyTaskEncoder(
        tokenizer_path=tokenizer_path,
        seq_length=66536,
        sensitive_words_path=None,
    ),
    batch_size=1,
    packing_buffer_size=None,
    shuffle_buffer_size=None,
    max_samples_per_sequence=None,
    worker_config=worker_config,
)

train_dataloader = get_loader(train_ds, worker_config=worker_config,)

rank=0, worker=0: sample_range=[0, 7895] in 1 slices, sum(count)=7895: indexes=[0, 7895] slices=[train-chunk-0.tar[0(start), 7895(end)]]
rank=0, worker=0: sample_range=[0, 7619] in 1 slices, sum(count)=7619: indexes=[0, 7619] slices=[train-chunk-0.tar[0(start), 7619(end)]]
rank=0, worker=0: sample_range=[0, 24050] in 1 slices, sum(count)=24050: indexes=[0, 24050] slices=[train-chunk-0.tar[0(start), 24050(end)]]


/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/miniconda3/envs/megatron-next/lib/python3.12/site-packages/megatron/energon/loader.py:108: FutureWarning: Passing a worker_config to get_loader() is deprecated and will have no effect.
  warn_deprecated(


In [5]:
dd = iter(train_dataloader)

In [6]:
import json

count_map = {}

with open("/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/code/Megatron-Next/model_dir/pulse_v18_3_30b_a3b_gemini_bf16/qua/pulse_v19_32b_gemini_train.jsonl", "w", encoding="utf8") as f:
    for _ in tqdm(range(2048)):
        item = next(dd)
        
        kk = item['__key__'][0].split("/")[-1].split("-train-")[0]
        if kk not in count_map:
            count_map[kk] = 0
        
        count_map[kk] += 1
              
        item = {
            "input_ids": item['input_ids'].tolist()[0],
            "labels": item['labels'].tolist()[0],
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        

            
        

100%|██████████| 2048/2048 [06:22<00:00,  5.35it/s]


In [8]:
count_list = sorted(count_map.items(), key=lambda x:-x[1])
[(k,v/2048)for k,v in count_list]

[('healthbench_ex-same-language-think', 0.3603515625),
 ('healthbench_base-same-language-think', 0.34716796875),
 ('global_health-same-language-think', 0.29248046875)]

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [10]:
tokenizer("<|im_start|>assistant\n<think>\n").input_ids 

[151644, 77091, 198, 151667, 198]

In [11]:
tokenizer("<|im_start|>assistant\n<ggthink>\n").input_ids 

[151644, 77091, 198, 27, 14398, 26865, 397]

In [12]:
item = next(dd)
# print(item['__key__'][0].split("/")[-1].split("-train-")[0])
print(tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100]))

<|im_start|>system
**Role Definition:**
You are an expert, clinically rigorous AI medical consultant. Your goal is to provide comprehensive, safe, and actionable medical guidance. You must simulate the logic of an experienced clinician, prioritizing patient safety, precise triage, holistic care, and patient empowerment. You avoid overconfidence and acknowledge the limitations of AI in specific clinical nuances.

**INTERNAL REASONING PROTOCOL:**
**You must use English for your internal thinking process and chain-of-thought.** This ensures the highest level of logical consistency and medical accuracy.
1.  **Context Check:** Identify missing critical parameters (Age, Sex, Location, History, Meds).
2.  **Calculation Verification:** If any scoring system (e.g., MEWS, APGAR) or dosage calculation is required, perform a step-by-step verification of the criteria and math in your internal monologue to prevent scoring errors.
3.  **Uncertainty Assessment:** explicitly evaluate if the answer requ

In [13]:
print(tokenizer.decode([item for item in item['labels'].tolist()[0] if item != -100]))

<think>
The user is a 35-year-old female, G3P1 (1 term birth, 1 miscarriage at 5 weeks). Currently pregnant at 6w+6d (based on LMP April 25).
    *   LMP: April 25.
    *   Coitus: Day 14 (May 8).
    *   Beta-hCG levels:
        *   4w+5d: 1,311
        *   5w+0d: 1,932 (slow rise)
        *   5w+6d: 1,903 (plateau/slight drop)
        *   6w+1d: 1,978 (plateau)
    *   Ultrasound at 6w+1d: CRL 2.7 mm + Fetal Heart Rate (FHR) present.
    *   Symptoms: Dark brown discharge/spotting.
    *   Ultrasound at 6w+5d: CRL 4.8 mm + FHR present.
    *   Medication: Folic acid.
    *   Concern: Low/stagnant Beta-hCG vs. embryo growth and heartbeat. Potential for a successful pregnancy?

    *   *Beta-hCG:* In a healthy pregnancy, Beta-hCG typically doubles every 48-72 hours in the early stages until it reaches about 6,000-10,000 mIU/mL. The user's levels have plateaued around 1,900 for over a week. This is highly abnormal and usually associated with miscarriage or ectopic pregnancy (though an i